In [13]:
DATA_DIR = "./GeneratedLabelledFlows/TrafficLabelling_"  # <-- change this
OUT_PATH = "preprocessed.parquet"
LABEL_COL = "Label"

## Imports

In [14]:
import glob
import os

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Load all CSVs

In [15]:
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

print(f"Found {len(csv_files)} CSV files")
dfs = []
for f in csv_files:
    print(f"  loading {os.path.basename(f)}")
    try:
        df_part = pd.read_csv(f, low_memory=False)
    except UnicodeDecodeError:
        df_part = pd.read_csv(f, low_memory=False, encoding="latin1")
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)
del dfs
print(f"Combined shape: {df.shape}")

Found 8 CSV files
  loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  loading Friday-WorkingHours-Morning.pcap_ISCX.csv
  loading Monday-WorkingHours.pcap_ISCX.csv
  loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  loading Tuesday-WorkingHours.pcap_ISCX.csv
  loading Wednesday-workingHours.pcap_ISCX.csv
Combined shape: (3119345, 85)


## Clean column names

In [16]:
df.columns = [c.strip() for c in df.columns]

dup_cols = df.columns[df.columns.duplicated()].tolist()
if dup_cols:
    print("Dropping duplicate columns:", dup_cols)
    df = df.loc[:, ~df.columns.duplicated()]

print(df.columns.tolist())

['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH 

## Identify ID, label, and feature columns

In [17]:
id_cols = ["Source IP", "Destination IP", "Source Port", "Destination Port",
           "Protocol", "Timestamp"]
id_cols = [c for c in id_cols if c in df.columns]

missing = {"Source IP", "Destination IP"} - set(id_cols)
if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        "Use GeneratedLabelledFlows, not MachineLearningCSV"
    )

exclude = set(id_cols + [LABEL_COL, "Flow ID"])
feature_cols = [c for c in df.columns
                if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
print(f"Using {len(feature_cols)} numeric feature columns")

Using 77 numeric feature columns


## Handle NaN / inf

In [18]:
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
n_before = len(df)
df = df.dropna(subset=feature_cols)
print(f"Dropped {n_before - len(df)} rows with NaN/inf values")
print(f"Remaining rows: {len(df)}")

Dropped 291469 rows with NaN/inf values
Remaining rows: 2827876


## Encode labels

In [19]:
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

df["binary_label"] = (df[LABEL_COL] != "BENIGN").astype(int)

label_encoder = LabelEncoder()
df["multiclass_label"] = label_encoder.fit_transform(df[LABEL_COL])

print("Binary distribution:")
print(df["binary_label"].value_counts(), "\n")
print("Multiclass distribution:")
print(df[LABEL_COL].value_counts())

Binary distribution:
binary_label
0    2271320
1     556556
Name: count, dtype: int64 

Multiclass distribution:
Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
